In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    temperature=0.1,
)


def add_message(memory, input, output):
    memory.save_context({"input": input}, {"output": output})


def get_history(memory):
    return memory.load_memory_variables({})

In [ ]:
from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(return_messages=True)

memory.save_context({"input": "Hi!"}, {"output": "Hello!"})

memory.load_memory_variables({})

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(return_messages=True, k=4)


add_message(memory, "1", "1")
add_message(memory, "2", "2")
add_message(memory, "3", "3")
add_message(memory, "4", "4")
add_message(memory, "5", "5")
add_message(memory, "6", "6")

memory.load_memory_variables({})

In [ ]:
from langchain.memory import ConversationSummaryMemory

memory = ConversationSummaryMemory(llm=llm)

add_message(
    memory, "Hi I'm Chanwoo, I live in south korea", "Hello Chanwoo, nice to meet you!"
)
add_message(memory, "South Korea is so pretty", "I with could visit there someday")
get_history(memory)

In [ ]:
from langchain.memory import ConversationSummaryBufferMemory
from langchain.chains.llm import LLMChain
from langchain.prompts import ChatPromptTemplate, MessagesPlaceholder

memory = ConversationSummaryBufferMemory(
    llm=llm, max_token_limit=120, memory_key="chat_history", return_messages=True
)


prompt = ChatPromptTemplate(
    [
        ("system", "You are a helpful AI talking to a human."),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{question}"),
    ]
)

chain = LLMChain(
    llm=llm,
    memory=memory,
    prompt=prompt,
    verbose=True,
)

chain.predict(question="My name is Chanwoo")

In [ ]:
chain.predict(question="I live in South Korea")

In [ ]:
chain.predict(question="What is my name")

In [ ]:
from langchain.schema.runnable import RunnablePassthrough

memory = ConversationSummaryBufferMemory(
    llm=llm, max_token_limit=120, memory_key="chat_history", return_messages=True
)


def load_memory(input):
    return memory.load_memory_variables({})["chat_history"]


chain = RunnablePassthrough.assign(chat_history=load_memory) | prompt | llm


def invoke_chain(question):
    result = chain.invoke(
        {
            "question": question,
        }
    )
    memory.save_context({"input": question}, {"output": result.content})
    print(result)


invoke_chain("My name is Chanwoo")

content='Nice to meet you, Chanwoo! How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 26, 'total_tokens': 42, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-c26da9df-4e1e-46c4-bda5-f5f97dfe4217-0' usage_metadata={'input_tokens': 26, 'output_tokens': 16, 'total_tokens': 42, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [56]:
invoke_chain("What is my name?")

content='Your name is Chanwoo.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 54, 'total_tokens': 61, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run-7f6065d9-0234-4ae6-8292-afdfbe562f70-0' usage_metadata={'input_tokens': 54, 'output_tokens': 7, 'total_tokens': 61, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
